In [2]:
# # Install required packages
# !pip install pypdf==5.6.0
# !pip install PyMuPDF==1.26.1
# !pip install python-dotenv==1.1.0
# !pip install langchain-community==0.3.25
# !pip install langchain_openai==0.3.23
# !pip install rank_bm25==0.2.2
# !pip install faiss-cpu==1.11.0
# !pip install deepeval==3.1.0

In [3]:
# Clone the repository to access helper functions and evaluation modules
!git clone https://github.com/NirDiamant/RAG_TECHNIQUES.git
import sys
sys.path.append('RAG_TECHNIQUES')

# If you need to run with the latest data
# !cp -r RAG_TECHNIQUES/data .

Cloning into 'RAG_TECHNIQUES'...
remote: Enumerating objects: 2400, done.
remote: Counting objects: 100% (190/190), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 2400 (delta 153), reused 96 (delta 96), pack-reused 2210 (from 3)
Receiving objects: 100% (2400/2400), 41.58 MiB | 4.63 MiB/s, done.
Resolving deltas: 100% (1584/1584), done.


In [6]:
import os
import sys
from dotenv import load_dotenv
#from google.colab import userdata



import sys
from pathlib import Path

# Setup paths
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from utils.env_loader import load_environment
load_environment()


# Original path append replaced for Colab compatibility

from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from helper_functions import (EmbeddingProvider,
                              retrieve_context_per_question,
                              replace_t_with_space,
                              get_langchain_embedding_provider,
                              show_context)

from evaluation.evalute_rag import evaluate_rag

from langchain.vectorstores import FAISS

In [ ]:
# Download required data files
import os
os.makedirs('data', exist_ok=True)

# Download the PDF document used in this notebook
#!wget -O data/Understanding_Climate_Change.pdf https://raw.githubusercontent.com/NirDiamant/RAG_TECHNIQUES/main/data/Understanding_Climate_Change.pdf
#!wget -O data/Understanding_Climate_Change.pdf https://raw.githubusercontent.com/NirDiamant/RAG_TECHNIQUES/main/data/Understanding_Climate_Change.pdf
#--

zsh:1: command not found: wget
zsh:1: command not found: wget


In [8]:
path = "data/Understanding_Climate_Change.pdf"

In [9]:
def encode_pdf(path, chunk_size=1000, chunk_overlap=200):
    """
    Encodes a PDF book into a vector store using OpenAI embeddings.

    Args:
        path: The path to the PDF file.
        chunk_size: The desired size of each text chunk.
        chunk_overlap: The amount of overlap between consecutive chunks.

    Returns:
        A FAISS vector store containing the encoded book content.
    """

    # Load PDF documents
    loader = PyPDFLoader(path)
    documents = loader.load()

    # Split documents into chunks
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap, length_function=len
    )
    texts = text_splitter.split_documents(documents)
    cleaned_texts = replace_t_with_space(texts)

    # Create embeddings (Tested with OpenAI and Amazon Bedrock)
    embeddings = get_langchain_embedding_provider(EmbeddingProvider.OPENAI)
    #embeddings = get_langchain_embedding_provider(EmbeddingProvider.AMAZON_BEDROCK)

    # Create vector store
    vectorstore = FAISS.from_documents(cleaned_texts, embeddings)

    return vectorstore

In [10]:
chunks_vector_store = encode_pdf(path, chunk_size=1000, chunk_overlap=200)

In [11]:
chunks_query_retriever = chunks_vector_store.as_retriever(search_kwargs={"k": 2})

In [12]:
test_query = "What is the main cause of climate change?"
context = retrieve_context_per_question(test_query, chunks_query_retriever)
show_context(context)

/Users/kanderaolaxminarasimharao/Downloads/MyAgents-Git/MyAgents/notebooks/RAG_TECHNIQUES/helper_functions.py:143: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  docs = chunks_query_retriever.get_relevant_documents(question)


: 